# MeritKV exact-prefix splice validation on Gemma-4 E2B and Qwen2.5-1.5B

This notebook validates a token-identical KV-cache crop-and-splice path on `google/gemma-4-E2B-it` and `Qwen/Qwen2.5-1.5B-Instruct`. It compares full recomputation, native prefix-cache continuation, and continuation from a longer cache cropped to the identical prefix. It records per-layer KV agreement, aligned suffix-logit agreement, and teacher-forced greedy-token agreement.

Run all cells on a Colab **T4 GPU**. Do not change tolerances after seeing the result. A failed validation is still a valid, reportable outcome.

In [ ]:
from google.colab import files
from pathlib import Path
import io, shutil, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(zip_names) == 1, f'Upload exactly one ZIP; received: {list(uploaded)}'
extract_root = Path('/content/meritkv_splice_bundle')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
    print('Archive members:', *archive.namelist(), sep='\n- ')
    for member in archive.infolist():
        normalized = member.filename.replace('\\', '/')
        parts = [part for part in normalized.split('/') if part not in ('', '.')]
        assert '..' not in parts, f'Unsafe archive member: {member.filename}'
        if not parts:
            continue
        target = extract_root.joinpath(*parts)
        if member.is_dir() or normalized.endswith('/'):
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
runner_matches = list(extract_root.rglob('validate_exact_splice.py'))
assert len(runner_matches) == 1, f'Expected one runner, found {runner_matches}'
ROOT = runner_matches[0].parent
print('Bundle root:', ROOT)
print('Files:', *sorted(p.name for p in ROOT.iterdir()), sep='\n- ')

## Install the recorded software stack

This installs the Transformers-side dependencies only. It intentionally keeps Colab's CUDA-enabled PyTorch build.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(ROOT / 'requirements-colab.txt')
])

In [ ]:
import torch, transformers, accelerate, huggingface_hub, safetensors
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=True).stdout)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name, f'This package was requested for a T4; current GPU is {gpu_name!r}'
print({
    'python': sys.version,
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'accelerate': accelerate.__version__,
    'huggingface_hub': huggingface_hub.__version__,
    'safetensors': safetensors.__version__,
    'gpu': gpu_name,
})

## Authenticate and freeze the exact model revision

Accept the checkpoint's Hugging Face access terms first. The cell uses a Colab secret named `HF_TOKEN` if available; otherwise it opens the standard login prompt.

In [ ]:
from huggingface_hub import login, model_info
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)
MODEL_IDS = {
    'gemma4_e2b': 'google/gemma-4-E2B-it',
    'qwen25_15b': 'Qwen/Qwen2.5-1.5B-Instruct',
}
MODEL_REVISIONS = {key: model_info(model_id).sha for key, model_id in MODEL_IDS.items()}
for key, model_id in MODEL_IDS.items():
    print(f'{key}: {model_id} @ {MODEL_REVISIONS[key]}')

## Self-test the validation harness

This tiny CPU test checks the comparison machinery before downloading either evaluated checkpoint. It is not paper evidence.

In [ ]:
OUTPUT = Path('/content/meritkv_exact_splice_validation_outputs')
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True)
SELF_TEST_OUTPUT = OUTPUT / 'self_test'
SELF_TEST_OUTPUT.mkdir(parents=True)
self_test_cmd = [
    sys.executable, str(ROOT / 'validate_exact_splice.py'),
    '--tiny-model', 'tiny-qwen2',
    '--device', 'cpu', '--dtype', 'float32',
    '--attn-implementation', 'eager',
    '--sequence-length', '24', '--shared-ratios', '0.5',
    '--samples', '1', '--max-new-tokens', '4',
    '--atol', '1e-5', '--rtol', '1e-5',
    '--output', str(SELF_TEST_OUTPUT / 'harness_self_test.json'),
]
print(' '.join(self_test_cmd))
subprocess.run(self_test_cmd, check=True)

## Define the checkpoint validation run

Return code `2` means one or more strict checks failed; the JSON is retained and the notebook continues so that you can return the evidence. Any other nonzero return code is an execution failure.

In [ ]:
import json, shlex
command_log = []

def run_checkpoint_validation(model_key, attention_implementation):
    model_id = MODEL_IDS[model_key]
    model_revision = MODEL_REVISIONS[model_key]
    output_json = OUTPUT / f'{model_key}_t4_f16_{attention_implementation}.json'
    cmd = [
        sys.executable, str(ROOT / 'validate_exact_splice.py'),
        '--model-id', model_id, '--revision', model_revision,
        '--device', 'cuda:0', '--dtype', 'float16',
        '--attn-implementation', attention_implementation,
        '--sequence-length', '128',
        '--shared-ratios', '0.5', '0.75',
        '--samples', '4', '--max-new-tokens', '8',
        '--seed', '42', '--atol', '0.001', '--rtol', '0.001',
        '--output', str(output_json),
    ]
    rendered = ' '.join(shlex.quote(part) for part in cmd)
    command_log.append(rendered)
    (OUTPUT / 'command_log.txt').write_text('\n'.join(command_log) + '\n', encoding='utf-8')
    print(rendered, flush=True)
    log_lines = []
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log_lines.append(line)
    return_code = process.wait()
    (OUTPUT / f'{model_key}_t4_f16_{attention_implementation}.log').write_text(''.join(log_lines), encoding='utf-8')
    if return_code not in (0, 2):
        raise RuntimeError(f'Execution failed with return code {return_code}')
    report = json.loads(output_json.read_text(encoding='utf-8'))
    print(json.dumps(report['summary'], indent=2))
    return report


## Run 1: Gemma-4 E2B, eager attention

Treat this as the primary, easiest-to-interpret correctness check.

In [ ]:
gemma_eager_report = run_checkpoint_validation('gemma4_e2b', 'eager')

## Run 2: Gemma-4 E2B, SDPA attention

This separately checks the optimized PyTorch attention implementation on the same cases.

In [ ]:
gemma_sdpa_report = run_checkpoint_validation('gemma4_e2b', 'sdpa')

## Run 3: Qwen2.5-1.5B, eager attention

This is the primary corrected-path check for the Qwen configuration that produced the paper's excluded float16 splice result.

In [ ]:
qwen_eager_report = run_checkpoint_validation('qwen25_15b', 'eager')

## Run 4: Qwen2.5-1.5B, SDPA attention

This separately checks the optimized PyTorch attention implementation on the same deterministic Qwen cases.

In [ ]:
qwen_sdpa_report = run_checkpoint_validation('qwen25_15b', 'sdpa')

## Record environment, summarize, checksum, and download

The downloaded ZIP is the complete result to return. Do not manually edit its JSON or summary.

In [ ]:
revision_lines = [f'{key}  {MODEL_IDS[key]}  {MODEL_REVISIONS[key]}' for key in MODEL_IDS]
(OUTPUT / 'model_revisions.txt').write_text('\n'.join(revision_lines) + '\n', encoding='utf-8')
(OUTPUT / 'nvidia_smi.txt').write_text(
    subprocess.run(['nvidia-smi'], text=True, capture_output=True, check=True).stdout,
    encoding='utf-8',
)
(OUTPUT / 'pip_freeze.txt').write_text(
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], text=True, capture_output=True, check=True).stdout,
    encoding='utf-8',
)
summary_cmd = [
    sys.executable, str(ROOT / 'summarize_validation.py'), str(OUTPUT),
    '--output-json', str(OUTPUT / 'combined_summary.json'),
    '--output-md', str(OUTPUT / 'VALIDATION_SUMMARY.md'),
]
subprocess.run(summary_cmd, check=True)
print((OUTPUT / 'VALIDATION_SUMMARY.md').read_text(encoding='utf-8'))

In [ ]:
import hashlib
manifest_lines = []
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file() and p.name != 'MANIFEST_SHA256.txt'):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest_lines.append(f'{digest}  {path.relative_to(OUTPUT).as_posix()}')
(OUTPUT / 'MANIFEST_SHA256.txt').write_text('\n'.join(manifest_lines) + '\n', encoding='utf-8')
archive_base = Path('/content/meritkv_exact_splice_validation_outputs')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT))
print('Archive:', archive_path, archive_path.stat().st_size, 'bytes')
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
    print(*archive.namelist(), sep='\n')
files.download(str(archive_path))

## What to return

Send back `meritkv_exact_splice_validation_outputs.zip` exactly as downloaded. It contains separate eager and SDPA reports for Gemma-4 E2B and Qwen2.5-1.5B. The decisive fields are `all_strict_pass`, the per-layer prefix-cache comparisons, aligned suffix-logit comparisons, and native/spliced greedy-token agreement.